# 🔌 Actividad 2 — Régimen Transitorio en DC (Circuitos RL y RC)

**Curso:** Física: Electricidad y Magnetismo  
**Programa:** Ingeniería Informática (Modalidad Virtual) — Tercer semestre  
**Tema 3:** Análisis de circuitos con bobinas y condensadores en corriente directa

---

## 🎯 Bienvenida

En la **Actividad 1** trabajaste con circuitos formados solo por **resistencias**: la corriente respondía instantáneamente al voltaje, sin retraso.

Ahora vamos a estudiar dos componentes nuevos que tienen una característica fascinante: **no responden de inmediato**, sino que la corriente o el voltaje **evolucionan en el tiempo** siguiendo curvas exponenciales.

| Componente | Símbolo | ¿Qué hace? |
|---|---|---|
| **Bobina** | L (henrios, H) | Se opone a los **cambios de corriente**. Cuando enciendes el circuito, la corriente crece poco a poco. |
| **Condensador** | C (faradios, F) | **Almacena carga eléctrica**. Se carga y se descarga progresivamente. |

> 🧭 La actividad tiene **3 partes**:
> - **Parte A** — Circuito RL: cómo crece la corriente hacia su valor límite.  
> - **Parte B** — Circuito RC: cómo se carga un condensador.  
> - **Parte C** — Circuito RC: cómo se descarga un condensador.

---

## 🧭 Concepto clave: la constante de tiempo τ

La **velocidad** con la que cambian la corriente o el voltaje se mide con un parámetro llamado **constante de tiempo (τ, "tau")**, medida en segundos.

**Regla universal:** después de un tiempo t = τ, la magnitud que está creciendo ha alcanzado el **63%** de su valor final. Después de t = 5τ, ya está prácticamente en el 99% (el régimen transitorio "terminó").

- En **circuitos RL** (resistencia + bobina): τ = L / R
- En **circuitos RC** (resistencia + condensador): τ = R · C

> 💡 En cada parte vas a tener un **slider de tiempo** para "viajar" sobre la curva y leer el valor de la corriente o el voltaje en cualquier instante. Úsalo para **encontrar τ visualmente**: el instante donde la curva alcanza el 63% del valor final.


## ⚠️ Importante — Cómo ejecutar este notebook

Este notebook está dividido en **celdas** que debes ejecutar **en orden, de arriba hacia abajo**:

1. **Paso 0** — Configuración inicial (importa librerías, prepara el sistema).
2. **Paso 1** — Te identificas con tu correo institucional.
3. **Partes A, B, C** — Trabajas con cada simulador y respondes las preguntas.
4. **Paso final** — Envías los datos al docente.

> 🔄 **Si reinicias el kernel** o vuelves al notebook después de cerrarlo, debes **volver a ejecutar las celdas en orden** desde el Paso 0. Cada simulador verifica automáticamente que el sistema esté listo y te avisará claramente si falta algo.

> ▶️ Para ejecutar una celda, haz clic sobre ella y presiona `Shift + Enter`, o usa el botón ▶ a la izquierda.


## 🛠️ Paso 0 — Configuración inicial

**▶️ Ejecuta la siguiente celda** para cargar el sistema. Toma unos segundos.


In [1]:
# ═══ Configuración e instrumentación — Actividad 2 ═══
import sys, subprocess, json, time, uuid, datetime

for pkg in ["ipywidgets", "matplotlib", "numpy"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#FAFAFA",
    "axes.edgecolor": "#CCCCCC",
    "axes.labelcolor": "#333333",
    "xtick.color": "#666666",
    "ytick.color": "#666666",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})


# ─── CAPA DE INSTRUMENTACIÓN (idéntica a A1) ──────────────────────────
class Tracker:
    def __init__(self):
        self.session_id  = str(uuid.uuid4())[:12]
        self.t_start     = time.time()
        self.events      = []
        self.answers     = []
        self.outcomes    = []
        self.last_action = self.t_start
        self.est_id      = None
        self.activity    = "A2_TRANSITORIO"

    def _now(self):
        return round(time.time() - self.t_start, 3)

    def _gap(self):
        gap = round(time.time() - self.last_action, 3)
        self.last_action = time.time()
        return gap

    def widget_event(self, part, widget_name, old, new, target=None):
        self.events.append({
            "session_id": self.session_id,
            "est_id":     self.est_id,
            "activity":   self.activity,
            "type":       "widget_event",
            "part":       part,
            "widget":     widget_name,
            "old_value":  old,
            "new_value":  new,
            "target":     target,
            "distance_to_target": (None if target is None else round(abs(new - target), 6)),
            "params":     None,
            "results":    None,
            "error_pct":  None,
            "t_relative": self._now(),
            "gap_since_last": self._gap(),
            "timestamp":  datetime.datetime.utcnow().isoformat(),
        })

    def simulation(self, part, params, results, error_pct=None):
        self.events.append({
            "session_id": self.session_id,
            "est_id":     self.est_id,
            "activity":   self.activity,
            "type":       "simulation",
            "part":       part,
            "widget":     None,
            "old_value":  None,
            "new_value":  None,
            "target":     None,
            "distance_to_target": None,
            "params":     json.dumps(params),
            "results":    json.dumps(results),
            "error_pct":  error_pct,
            "t_relative": self._now(),
            "gap_since_last": None,
            "timestamp":  datetime.datetime.utcnow().isoformat(),
        })

    def answer(self, part, qid, value, correct=None, expected=None):
        self.answers.append({
            "session_id": self.session_id,
            "est_id":     self.est_id,
            "activity":   self.activity,
            "part":       part,
            "qid":        qid,
            "value":      str(value),
            "expected":   None if expected is None else str(expected),
            "is_correct": None if correct is None else bool(correct),
            "t_relative": self._now(),
            "timestamp":  datetime.datetime.utcnow().isoformat(),
        })

    def outcome(self, part, score, completed=True):
        self.outcomes.append({
            "session_id":   self.session_id,
            "est_id":       self.est_id,
            "activity":     self.activity,
            "part":         part,
            "score":        score,
            "completed":    completed,
            "duration_sec": self._now(),
            "n_widget_events": sum(1 for e in self.events
                                   if e["part"] == part and e["type"] == "widget_event"),
            "n_simulations":   sum(1 for e in self.events
                                   if e["part"] == part and e["type"] == "simulation"),
            "timestamp":    datetime.datetime.utcnow().isoformat(),
        })

    def summary(self):
        return {
            "session_id":  self.session_id,
            "duration_min": round((time.time() - self.t_start) / 60, 1),
            "n_widget_events": sum(1 for e in self.events if e["type"] == "widget_event"),
            "n_simulations":   sum(1 for e in self.events if e["type"] == "simulation"),
            "n_answers":   len(self.answers),
            "n_outcomes":  len(self.outcomes),
        }


TR = Tracker()


# ─── HELPERS DE VISUALIZACIÓN (idénticos a A1) ────────────────────────
def metric_card(label, value, unit, color="#1D4ED8"):
    return f"""
    <div style="display: inline-block; background: white;
                border-left: 5px solid {color}; padding: 10px 18px;
                margin: 4px; border-radius: 6px;
                box-shadow: 0 1px 3px rgba(0,0,0,0.08); min-width: 130px;">
        <div style="font-size: 11px; color: #666; text-transform: uppercase;
                    letter-spacing: 0.5px; font-weight: 600;">{label}</div>
        <div style="font-size: 22px; color: {color}; font-weight: 700;
                    margin-top: 2px;">
            {value} <span style="font-size: 13px; color: #999;
                                 font-weight: 400;">{unit}</span>
        </div>
    </div>
    """


def status_badge(text, kind="info"):
    colors = {
        "info":    ("#DBEAFE", "#1D4ED8"),
        "success": ("#DCFCE7", "#15803D"),
        "warning": ("#FEF3C7", "#D97706"),
        "error":   ("#FEE2E2", "#B91C1C"),
    }
    bg, fg = colors.get(kind, colors["info"])
    return f"""
    <span style="display: inline-block; background: {bg}; color: {fg};
                 padding: 3px 10px; border-radius: 12px;
                 font-size: 11px; font-weight: 600; margin: 2px;">{text}</span>
    """


# ─── HELPERS DE DIBUJO DE COMPONENTES ─────────────────────────────────
def draw_resistor_h(ax, x0, y0, length, label, value, color, label_above=True):
    """Resistor zigzag horizontal de (x0,y0) a (x0+length, y0).
       Etiquetas: label (nombre) en posición prominente (más alejada del componente),
       value (valor numérico) cerca del componente."""
    n_zigs = 7
    zx = np.linspace(x0, x0 + length, n_zigs * 2 + 2)
    zy = np.full_like(zx, y0, dtype=float)
    amp = 0.2
    for i in range(1, len(zy) - 1):
        zy[i] = y0 + amp * ((-1) ** i)
    ax.plot(zx, zy, color=color, lw=2.5)
    if label_above:
        # label (nombre, más grande) arriba del todo
        ax.text(x0 + length/2, y0 + 0.95, label, ha="center",
                fontsize=11, fontweight="bold", color=color)
        # value (valor) justo debajo del nombre
        ax.text(x0 + length/2, y0 + 0.55, value, ha="center",
                fontsize=8, fontweight="bold", color=color)
    else:
        ax.text(x0 + length/2, y0 - 0.95, label, ha="center",
                fontsize=11, fontweight="bold", color=color)
        ax.text(x0 + length/2, y0 - 0.55, value, ha="center",
                fontsize=8, fontweight="bold", color=color)


def draw_inductor_h(ax, x0, y0, length, label, value, color):
    """Bobina como serie de arcos semicirculares horizontales.
       Los arcos suben hasta y0 + seg_len/2; las etiquetas van más arriba."""
    n_arcs = 4
    seg_len = length / n_arcs
    for i in range(n_arcs):
        cx = x0 + (i + 0.5) * seg_len
        theta = np.linspace(0, np.pi, 30)
        ax.plot(cx + (seg_len/2) * np.cos(theta),
                y0 + (seg_len/2) * np.sin(theta),
                color=color, lw=2.5)
    # label (nombre) en posición prominente (arriba del todo)
    ax.text(x0 + length/2, y0 + 1.55, label, ha="center",
            fontsize=11, fontweight="bold", color=color)
    # value (valor) justo debajo del nombre, todavía arriba de los arcos
    ax.text(x0 + length/2, y0 + 1.15, value, ha="center",
            fontsize=8, fontweight="bold", color=color)


def draw_capacitor_v(ax, x, y_center, label, value, color):
    """Condensador vertical (dos placas paralelas horizontales)."""
    plate_w = 0.7
    gap = 0.18
    # Placa superior
    ax.plot([x - plate_w/2, x + plate_w/2],
            [y_center + gap/2, y_center + gap/2], color=color, lw=3)
    # Placa inferior
    ax.plot([x - plate_w/2, x + plate_w/2],
            [y_center - gap/2, y_center - gap/2], color=color, lw=3)
    # Etiquetas a la derecha
    ax.text(x + plate_w/2 + 0.2, y_center + 0.15, label,
            ha="left", va="center", fontsize=10, fontweight="bold", color=color)
    ax.text(x + plate_w/2 + 0.2, y_center - 0.15, value,
            ha="left", va="center", fontsize=8, fontweight="bold", color=color)


def draw_battery(ax, x, y_center, voltage_label, color="#1D4ED8"):
    """Fuente DC como círculo con + y −."""
    ax.add_patch(Circle((x, y_center), 0.4, fill=True, facecolor="#DBEAFE",
                        edgecolor=color, lw=2))
    ax.text(x, y_center + 0.18, "+", ha="center", va="center",
            fontsize=12, fontweight="bold", color=color)
    ax.text(x, y_center - 0.18, "−", ha="center", va="center",
            fontsize=12, fontweight="bold", color=color)
    ax.text(x - 1.0, y_center, voltage_label, ha="center", va="center",
            fontsize=10, fontweight="bold", color=color,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor=color))


def draw_ground(ax, x, y_top):
    ax.plot([x, x], [y_top - 0.8, y_top], color="#333", lw=2)
    ax.plot([x - 0.4, x + 0.4], [y_top - 0.8, y_top - 0.8], color="#333", lw=3)
    ax.plot([x - 0.25, x + 0.25], [y_top - 1.0, y_top - 1.0], color="#333", lw=2)
    ax.plot([x - 0.1, x + 0.1], [y_top - 1.2, y_top - 1.2], color="#333", lw=1.5)


def ensure_runtime():
    """
    Verifica que el entorno esté listo (importaciones, Tracker, identificación).
    Cada celda de simulador la invoca al inicio para evitar NameError
    si el estudiante salta la celda de configuración o reinicia el kernel.
    """
    import sys
    g = globals()
    missing = []
    for name in ["widgets", "np", "plt", "Circle", "TR", "metric_card",
                 "status_badge", "draw_battery", "draw_resistor_h",
                 "draw_inductor_h", "draw_capacitor_v", "draw_ground"]:
        if name not in g:
            missing.append(name)
    if missing:
        from IPython.display import display, HTML
        display(HTML(f"""
        <div style="background:#FEE2E2; border-left:4px solid #B91C1C;
                    padding:14px; border-radius:6px;">
            <strong style="color:#7F1D1D; font-size:14px">
                ⚠️ Falta ejecutar la celda de configuración (Paso 0).
            </strong>
            <p style="margin:8px 0 0 0; color:#7F1D1D; font-size:13px">
                Variables faltantes: <code>{", ".join(missing)}</code>.<br>
                <strong>Solución:</strong> sube hasta el "Paso 0 — Configuración inicial"
                y ejecútalo (▶ o <code>Shift+Enter</code>). Luego también el Paso 1
                (Identifícate). Después regresa aquí y ejecuta de nuevo.
            </p>
        </div>
        """))
        raise RuntimeError("Configuración incompleta. Ejecuta primero el Paso 0.")
    if g["TR"].est_id is None:
        from IPython.display import display, HTML
        display(HTML("""
        <div style="background:#FEF3C7; border-left:4px solid #D97706;
                    padding:14px; border-radius:6px;">
            <strong style="color:#92400E">
                ⚠️ Aún no te has identificado (Paso 1).
            </strong>
            <p style="margin:6px 0 0 0; color:#78350F">
                Ve a la celda "Paso 1 — Identifícate" y registra tu correo
                institucional antes de continuar. Tus datos no se podrán enviar
                sin tu identificación.
            </p>
        </div>
        """))
        raise RuntimeError("Falta identificación (Paso 1).")


print("✅ Sistema cargado correctamente.")
print(f"   ID de sesión: {TR.session_id}")
print(f"   Hora de inicio: {datetime.datetime.now().strftime('%H:%M:%S')}")
print("   Funciones de simulador disponibles. Cada celda invocará ensure_runtime() "
      "para verificar el entorno automáticamente.")


✅ Sistema cargado correctamente.
   ID de sesión: dfeb6f24-783
   Hora de inicio: 18:05:13
   Funciones de simulador disponibles. Cada celda invocará ensure_runtime() para verificar el entorno automáticamente.


## 👤 Paso 1 — Identifícate

**▶️ Ejecuta la siguiente celda** y escribe tu correo institucional.


In [2]:
email_widget = widgets.Text(
    value="", placeholder="tu_correo@institucion.edu.co",
    description="Correo:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="450px"),
)
confirm_btn = widgets.Button(description="Confirmar", button_style="primary")
status_lbl  = widgets.HTML(value="")

def on_confirm(b):
    if "@" not in email_widget.value or "." not in email_widget.value:
        status_lbl.value = "<span style='color:#B91C1C'>⚠️ Ingresa un correo válido.</span>"
        return
    TR.est_id = email_widget.value.strip().lower()
    status_lbl.value = (
        f"<span style='color:#15803D; font-weight:600'>"
        f"✅ Identificado como {TR.est_id}. Ya puedes continuar con la Parte A.</span>"
    )
    email_widget.disabled = True
    confirm_btn.disabled  = True

confirm_btn.on_click(on_confirm)
display(widgets.VBox([email_widget, confirm_btn, status_lbl]))


---

## 🟢 Parte A — Circuito RL: corriente límite

### 📚 Concepto

Imagina que conectas una **bobina** (L) en serie con una **resistencia** (R) y una fuente DC (V), y de repente cierras un interruptor. ¿Qué pasa con la corriente?

A diferencia de un circuito puramente resistivo (donde I salta inmediatamente a V/R), la bobina **se opone a los cambios bruscos de corriente**. Por eso la corriente **crece poco a poco**, siguiendo una curva exponencial:

$$ i(t) = \frac{V}{R} \cdot \left(1 - e^{-t / \tau}\right) $$

donde la **constante de tiempo** es:

$$ \tau = \frac{L}{R} $$

### 🔑 Tres ideas clave

1. La **corriente límite** (cuando t → ∞) es:  $$ I_{max} = \frac{V}{R} $$

2. En el instante **t = τ**, la corriente vale **63% de I_max**.

3. En **t = 5τ** la corriente está prácticamente en su valor final (99%).

### 🎮 Simulador interactivo

Mueve los sliders para cambiar V, R, L y observa cómo varía la curva i(t).  
**Mueve el slider de tiempo** para "viajar" sobre la curva y leer el valor exacto de i en cualquier instante.

> 💡 **Reto:** ajusta L y R para que **τ = 1 ms** (1 milisegundo).


In [3]:
# ═══ Simulador Parte A — Circuito RL ═══
ensure_runtime()  # Verifica que la configuración esté lista


# Sliders de parámetros físicos
sliderA_V = widgets.FloatSlider(value=10.0, min=1, max=24, step=0.5,
                                description="V (V):", readout_format=".1f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))
sliderA_R = widgets.FloatSlider(value=100, min=10, max=1000, step=10,
                                description="R (Ω):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))
sliderA_L = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01,
                                description="L (H):", readout_format=".3f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))

# Slider de TIEMPO para medir i(t)
sliderA_t = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.05,
                                description="t (ms):", readout_format=".2f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="660px"))

metricsA = widgets.HTML()
plotA    = widgets.Output()


def draw_circuit_RL(V, R, L, ax):
    ax.clear()
    ax.set_xlim(0, 12); ax.set_ylim(0, 7)
    ax.set_aspect("equal"); ax.axis("off")

    # Cables
    ax.plot([1, 1], [2, 5.5], color="#333", lw=2)        # vertical izquierdo
    ax.plot([1, 4], [5.5, 5.5], color="#333", lw=2)      # superior izquierdo
    ax.plot([6, 8], [5.5, 5.5], color="#333", lw=2)      # superior medio
    ax.plot([10, 10.5], [5.5, 5.5], color="#333", lw=2)  # final superior
    ax.plot([10.5, 10.5], [2, 5.5], color="#333", lw=2)  # vertical derecho
    ax.plot([1, 10.5], [2, 2], color="#333", lw=2)       # inferior

    # Fuente
    draw_battery(ax, 1, 3.75, f"{V:.1f}V")

    # Resistor R en el cable superior (a la izquierda)
    draw_resistor_h(ax, 4, 5.5, 2, "R", f"{R:.0f}Ω", "#C2410C", label_above=True)

    # Bobina L en el cable superior (a la derecha)
    draw_inductor_h(ax, 8, 5.5, 2, "L", f"{L:.3f}H", "#7C3AED")

    # Flecha de corriente
    ax.annotate("", xy=(3, 5.5), xytext=(2, 5.5),
                arrowprops=dict(arrowstyle="->", color="#15803D", lw=2))
    ax.text(2.5, 5.95, "i(t)", ha="center",
            fontsize=10, fontweight="bold", color="#15803D")

    # Tierra
    draw_ground(ax, 10.5, 2)


def update_A(change=None):
    V = sliderA_V.value
    R = sliderA_R.value
    L = sliderA_L.value
    t_query_ms = sliderA_t.value
    t_query_s  = t_query_ms / 1000.0

    tau_s   = L / R                # segundos
    tau_ms  = tau_s * 1000          # milisegundos
    I_max   = V / R                 # amperios
    I_max_mA = I_max * 1000         # mA

    # i(t) en el instante consultado
    i_query_mA = I_max_mA * (1 - np.exp(-t_query_s / tau_s))

    # Detectar tipo de slider
    if change is not None and change.get("type") == "change" and change.get("name") == "value":
        widget_desc = change.owner.description.replace(":", "").strip()
        # ¿es el slider de tiempo o un slider físico?
        if widget_desc.startswith("t"):
            TR.widget_event(part="A", widget_name="t (slider tiempo)",
                            old=change["old"], new=change["new"])
        else:
            target = 1.0 if "L" in widget_desc or "R" in widget_desc else None
            TR.widget_event(part="A", widget_name=widget_desc,
                            old=change["old"], new=change["new"], target=target)
            TR.simulation(
                part="A",
                params={"V": V, "R": R, "L": L},
                results={"tau_ms": round(tau_ms, 4),
                         "I_max_mA": round(I_max_mA, 3)},
                error_pct=round(abs(tau_ms - 1.0) / 1.0 * 100, 2),
            )

    # Reto: τ = 1 ms
    target_msg = ""
    if abs(tau_ms - 1.0) < 0.05:
        target_msg = status_badge(f"🎯 ¡Lograste el reto: τ = {tau_ms:.3f} ms!", "success")

    # Métricas
    pct_at_t = (i_query_mA / I_max_mA * 100) if I_max_mA > 0 else 0
    metricsA.value = (
        metric_card("I_max (V/R)", f"{I_max_mA:.2f}", "mA", "#1D4ED8") +
        metric_card("τ = L/R", f"{tau_ms:.3f}", "ms (reto: 1.0)", "#C2410C") +
        metric_card("5τ (estable)", f"{5*tau_ms:.2f}", "ms", "#7C3AED") +
        metric_card(f"i(t={t_query_ms:.2f}ms)", f"{i_query_mA:.2f}", f"mA ({pct_at_t:.0f}% I_max)", "#15803D") +
        f"<div style='margin-top: 8px'>{target_msg}</div>"
    )

    with plotA:
        clear_output(wait=True)
        fig = plt.figure(figsize=(12, 6))
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.4], wspace=0.15)
        ax_circ = fig.add_subplot(gs[0])
        ax_curve = fig.add_subplot(gs[1])

        # Diagrama
        draw_circuit_RL(V, R, L, ax_circ)

        # Curva i(t)
        # Rango de t: hasta 6τ para que se vea la asintota
        t_max_plot_s = max(6 * tau_s, t_query_s * 1.2, 0.001)
        t_arr = np.linspace(0, t_max_plot_s, 500)
        i_arr_mA = I_max_mA * (1 - np.exp(-t_arr / tau_s))
        t_arr_ms = t_arr * 1000

        ax_curve.plot(t_arr_ms, i_arr_mA, color="#15803D", lw=2.5, label="i(t)")
        # Asíntota I_max
        ax_curve.axhline(I_max_mA, color="#1D4ED8", ls="--", lw=1.5,
                         label=f"I_max = {I_max_mA:.2f} mA")
        # Línea del 63%
        ax_curve.axhline(0.632 * I_max_mA, color="#C2410C", ls=":", lw=1.2,
                         label=f"63% = {0.632*I_max_mA:.2f} mA")
        # Marcador en τ
        ax_curve.axvline(tau_ms, color="#C2410C", ls=":", lw=1.2)
        ax_curve.text(tau_ms, I_max_mA * 0.05, f"τ={tau_ms:.3f}ms",
                      color="#C2410C", fontsize=9, fontweight="bold",
                      ha="left", rotation=0)

        # Marcador del slider de tiempo: punto rojo + línea vertical
        ax_curve.axvline(t_query_ms, color="#B91C1C", ls="-", lw=1.0, alpha=0.6)
        ax_curve.scatter([t_query_ms], [i_query_mA], color="#B91C1C", s=90,
                         zorder=5, edgecolors="white", linewidths=2)
        ax_curve.annotate(f"t={t_query_ms:.2f}ms\ni={i_query_mA:.2f}mA",
                          xy=(t_query_ms, i_query_mA),
                          xytext=(15, -25), textcoords="offset points",
                          fontsize=9, fontweight="bold", color="#B91C1C",
                          bbox=dict(boxstyle="round,pad=0.3",
                                    facecolor="#FEF2F2", edgecolor="#B91C1C"))

        ax_curve.set_xlabel("Tiempo t (ms)")
        ax_curve.set_ylabel("Corriente i (mA)")
        ax_curve.set_title("Crecimiento exponencial de la corriente en una bobina")
        ax_curve.legend(loc="lower right", fontsize=9)
        ax_curve.grid(True, alpha=0.3)
        ax_curve.set_xlim(0, t_max_plot_s * 1000)
        ax_curve.set_ylim(0, I_max_mA * 1.2 + 0.001)
        plt.show()


sliderA_V.observe(update_A, names="value")
sliderA_R.observe(update_A, names="value")
sliderA_L.observe(update_A, names="value")
sliderA_t.observe(update_A, names="value")
update_A()

display(widgets.VBox([
    widgets.HBox([sliderA_V, sliderA_R, sliderA_L]),
    sliderA_t,
    metricsA,
    plotA,
]))


/tmp/ipykernel_6642/3682259062.py:66: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":  datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_6642/3682259062.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":  datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_6642/3682259062.py:66: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":  datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_6642/3682259062.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and schedu

### 📝 Preguntas — Parte A


In [4]:
# Preguntas Parte A
qA1 = widgets.FloatText(description="A1 — Si V=12V, R=120Ω, L=60mH, ¿cuánto vale τ (en ms)?",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="600px"))
qA2 = widgets.FloatText(description="A2 — ¿Cuál es la corriente límite I_max (en mA)?",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="600px"))
qA3 = widgets.RadioButtons(
    options=["A los 5τ ya alcanzó prácticamente I_max",
             "Llega a I_max instantáneamente al cerrar el circuito",
             "Nunca alcanza I_max porque la bobina lo impide"],
    description="A3 — Sobre la corriente en un circuito RL:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)
submitA = widgets.Button(description="Enviar respuestas Parte A",
                         button_style="success",
                         layout=widgets.Layout(width="280px"))
feedA = widgets.HTML()

def on_submitA(b):
    # τ = L/R = 0.060/120 = 0.0005 s = 0.5 ms
    a1_ok = abs(qA1.value - 0.5) < 0.05
    # I_max = V/R = 12/120 = 0.1 A = 100 mA
    a2_ok = abs(qA2.value - 100.0) < 5.0
    a3_ok = qA3.value == "A los 5τ ya alcanzó prácticamente I_max"

    TR.answer("A", "qA1", qA1.value, correct=a1_ok, expected=0.5)
    TR.answer("A", "qA2", qA2.value, correct=a2_ok, expected=100.0)
    TR.answer("A", "qA3", qA3.value, correct=a3_ok,
              expected="A los 5τ ya alcanzó prácticamente I_max")

    score = sum([a1_ok, a2_ok, a3_ok]) / 3 * 100
    TR.outcome("A", score, completed=True)

    feedA.value = f"""
    <div style="background:#F0FDF4; border-left:4px solid #15803D;
                padding:14px; margin-top:10px; border-radius:6px;">
        <h4 style="margin:0; color:#14532D;">Puntaje Parte A: {score:.0f}/100</h4>
        <ul style="margin:8px 0; color:#333;">
            <li>A1 — τ esperado = 0.5 ms (= 60mH/120Ω)  {('✅' if a1_ok else '❌')}</li>
            <li>A2 — I_max esperada = 100 mA (= 12V/120Ω)  {('✅' if a2_ok else '❌')}</li>
            <li>A3 — {('✅' if a3_ok else '❌ — la corriente SÍ llega al valor límite, pero gradualmente')}</li>
        </ul>
    </div>
    """
    submitA.disabled = True

submitA.on_click(on_submitA)
display(widgets.VBox([qA1, qA2, qA3, submitA, feedA]))


/tmp/ipykernel_6642/3682259062.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":  datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_6642/3682259062.py:116: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":    datetime.datetime.utcnow().isoformat(),


---

## 🔵 Parte B — Circuito RC: carga del condensador

### 📚 Concepto

Ahora cambiamos la bobina por un **condensador** (C), que es un componente que **almacena carga eléctrica**. Cuando conectas un condensador descargado a una fuente DC a través de un resistor, el condensador empieza a "llenarse" de carga, y su voltaje crece exponencialmente desde 0 hasta el voltaje de la fuente:

$$ V_C(t) = V \cdot \left(1 - e^{-t / \tau}\right) $$

Pero ahora la **constante de tiempo** se calcula distinto:

$$ \tau = R \cdot C $$

### 🔄 Otro detalle interesante

Mientras el condensador se carga, la corriente que entra **decrece** exponencialmente desde un valor inicial alto hasta cero (cuando el condensador ya no acepta más carga):

$$ i(t) = \frac{V}{R} \cdot e^{-t / \tau} $$

Es decir: **el voltaje sube, la corriente baja**. Ambos son exponenciales con la misma τ pero "espejo" entre sí.

### 🎮 Simulador

Mueve los sliders y observa las dos curvas: V_C(t) y i(t).

> 💡 **Reto:** ajusta R y C para que el condensador alcance el **63% del voltaje final** justo en **t = 50 ms** (es decir, τ = 50 ms).


In [5]:
# ═══ Simulador Parte B — Circuito RC carga ═══
ensure_runtime()  # Verifica que la configuración esté lista


sliderB_V = widgets.FloatSlider(value=10.0, min=1, max=24, step=0.5,
                                description="V (V):", readout_format=".1f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))
sliderB_R = widgets.FloatSlider(value=1000, min=100, max=10000, step=100,
                                description="R (Ω):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))
sliderB_C = widgets.FloatSlider(value=50.0, min=1, max=500, step=1,
                                description="C (μF):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))

sliderB_t = widgets.FloatSlider(value=50.0, min=0.0, max=500.0, step=1.0,
                                description="t (ms):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="660px"))

metricsB = widgets.HTML()
plotB    = widgets.Output()


def draw_circuit_RC_charge(V, R, C, ax):
    ax.clear()
    ax.set_xlim(0, 13); ax.set_ylim(0, 7)
    ax.set_aspect("equal"); ax.axis("off")

    # Cables
    ax.plot([1, 1], [2, 5.5], color="#333", lw=2)
    ax.plot([1, 4], [5.5, 5.5], color="#333", lw=2)
    ax.plot([6, 9], [5.5, 5.5], color="#333", lw=2)
    ax.plot([9, 9], [2.6, 5.5], color="#333", lw=2)
    ax.plot([9, 9], [2, 2.4], color="#333", lw=2)
    ax.plot([1, 9], [2, 2], color="#333", lw=2)

    # Fuente
    draw_battery(ax, 1, 3.75, f"{V:.1f}V")

    # Resistor R
    draw_resistor_h(ax, 4, 5.5, 2, "R", f"{R:.0f}Ω", "#C2410C", label_above=True)

    # Condensador C (vertical, centrado en y=2.5)
    draw_capacitor_v(ax, 9, 2.5, "C", f"{C:.0f}μF", "#7C3AED")

    # Flecha de corriente
    ax.annotate("", xy=(3, 5.5), xytext=(2, 5.5),
                arrowprops=dict(arrowstyle="->", color="#15803D", lw=2))
    ax.text(2.5, 5.95, "i(t)", ha="center",
            fontsize=10, fontweight="bold", color="#15803D")

    # Indicador V_C(t) — colocado ARRIBA del condensador para evitar
    # solaparse con las etiquetas "C" y "{value}" que están a la derecha (x=9.55+)
    ax.annotate("V_C(t)", xy=(9, 2.65), xytext=(8.0, 4.2),
                ha="center", va="center", fontsize=9,
                fontweight="bold", color="#1D4ED8",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                          edgecolor="#1D4ED8"),
                arrowprops=dict(arrowstyle="->", color="#1D4ED8",
                                lw=1.2, connectionstyle="arc3,rad=-0.25"))


def update_B(change=None):
    V = sliderB_V.value
    R = sliderB_R.value
    C_uF = sliderB_C.value
    C_F  = C_uF * 1e-6
    t_query_ms = sliderB_t.value
    t_query_s  = t_query_ms / 1000.0

    tau_s  = R * C_F
    tau_ms = tau_s * 1000

    Vc_query = V * (1 - np.exp(-t_query_s / tau_s))
    i_query_mA = (V / R) * np.exp(-t_query_s / tau_s) * 1000

    if change is not None and change.get("type") == "change" and change.get("name") == "value":
        widget_desc = change.owner.description.replace(":", "").strip()
        if widget_desc.startswith("t"):
            TR.widget_event(part="B", widget_name="t (slider tiempo)",
                            old=change["old"], new=change["new"])
        else:
            target = 50.0 if "R" in widget_desc or "C" in widget_desc else None
            TR.widget_event(part="B", widget_name=widget_desc,
                            old=change["old"], new=change["new"], target=target)
            TR.simulation(
                part="B",
                params={"V": V, "R": R, "C_uF": C_uF},
                results={"tau_ms": round(tau_ms, 3), "Vc_at_5tau": round(0.99 * V, 3)},
                error_pct=round(abs(tau_ms - 50.0) / 50.0 * 100, 2),
            )

    target_msg = ""
    if abs(tau_ms - 50.0) < 1.0:
        target_msg = status_badge(f"🎯 ¡Lograste el reto: τ = {tau_ms:.2f} ms!", "success")

    pct_charged = (Vc_query / V * 100) if V > 0 else 0
    metricsB.value = (
        metric_card("V final", f"{V:.1f}", "V", "#1D4ED8") +
        metric_card("τ = R·C", f"{tau_ms:.2f}", "ms (reto: 50)", "#C2410C") +
        metric_card("5τ (estable)", f"{5*tau_ms:.0f}", "ms", "#7C3AED") +
        metric_card(f"V_C(t={t_query_ms:.0f}ms)", f"{Vc_query:.2f}", f"V ({pct_charged:.0f}% V)", "#15803D") +
        metric_card(f"i(t={t_query_ms:.0f}ms)", f"{i_query_mA:.2f}", "mA", "#B91C1C") +
        f"<div style='margin-top: 8px'>{target_msg}</div>"
    )

    with plotB:
        clear_output(wait=True)
        fig = plt.figure(figsize=(13, 6))
        gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], wspace=0.3)
        ax_circ = fig.add_subplot(gs[0])
        ax_v    = fig.add_subplot(gs[1])
        ax_i    = fig.add_subplot(gs[2])

        draw_circuit_RC_charge(V, R, C_uF, ax_circ)

        # Rango de tiempo
        t_max_plot_s = max(6 * tau_s, t_query_s * 1.2, 0.001)
        t_arr = np.linspace(0, t_max_plot_s, 500)
        Vc_arr = V * (1 - np.exp(-t_arr / tau_s))
        i_arr_mA = (V / R) * np.exp(-t_arr / tau_s) * 1000
        t_arr_ms = t_arr * 1000

        # Gráfica V_C
        ax_v.plot(t_arr_ms, Vc_arr, color="#1D4ED8", lw=2.5)
        ax_v.axhline(V, color="#1D4ED8", ls="--", lw=1.2,
                     label=f"V = {V:.1f} V")
        ax_v.axhline(0.632 * V, color="#C2410C", ls=":", lw=1.0,
                     label=f"63% = {0.632*V:.2f} V")
        ax_v.axvline(tau_ms, color="#C2410C", ls=":", lw=1.0)
        ax_v.text(tau_ms, V * 0.05, f"τ={tau_ms:.1f}ms",
                  color="#C2410C", fontsize=8, fontweight="bold", ha="left")
        ax_v.axvline(t_query_ms, color="#B91C1C", ls="-", lw=1.0, alpha=0.5)
        ax_v.scatter([t_query_ms], [Vc_query], color="#B91C1C", s=80,
                     zorder=5, edgecolors="white", linewidths=2)
        ax_v.set_xlabel("t (ms)"); ax_v.set_ylabel("V_C (V)")
        ax_v.set_title("Voltaje del condensador (sube)")
        ax_v.legend(loc="lower right", fontsize=8)
        ax_v.grid(True, alpha=0.3)
        ax_v.set_xlim(0, t_max_plot_s * 1000)
        ax_v.set_ylim(0, V * 1.2)

        # Gráfica i
        I0_mA = (V / R) * 1000
        ax_i.plot(t_arr_ms, i_arr_mA, color="#B91C1C", lw=2.5)
        ax_i.axhline(I0_mA, color="#B91C1C", ls="--", lw=1.2,
                     label=f"I₀ = {I0_mA:.2f} mA")
        ax_i.axhline(0.368 * I0_mA, color="#C2410C", ls=":", lw=1.0,
                     label=f"37% = {0.368*I0_mA:.2f} mA")
        ax_i.axvline(tau_ms, color="#C2410C", ls=":", lw=1.0)
        ax_i.axvline(t_query_ms, color="#B91C1C", ls="-", lw=1.0, alpha=0.5)
        ax_i.scatter([t_query_ms], [i_query_mA], color="#B91C1C", s=80,
                     zorder=5, edgecolors="white", linewidths=2)
        ax_i.set_xlabel("t (ms)"); ax_i.set_ylabel("i (mA)")
        ax_i.set_title("Corriente que entra al C (baja)")
        ax_i.legend(loc="upper right", fontsize=8)
        ax_i.grid(True, alpha=0.3)
        ax_i.set_xlim(0, t_max_plot_s * 1000)
        ax_i.set_ylim(0, I0_mA * 1.2 + 0.01)
        plt.show()


sliderB_V.observe(update_B, names="value")
sliderB_R.observe(update_B, names="value")
sliderB_C.observe(update_B, names="value")
sliderB_t.observe(update_B, names="value")
update_B()

display(widgets.VBox([
    widgets.HBox([sliderB_V, sliderB_R, sliderB_C]),
    sliderB_t,
    metricsB,
    plotB,
]))


In [6]:
# Preguntas Parte B
qB1 = widgets.FloatText(description="B1 — Si V=10V, R=2kΩ, C=100μF, ¿cuánto vale τ (en ms)?",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="600px"))
qB2 = widgets.FloatText(description="B2 — ¿Después de 5τ, qué voltaje aproximado tiene el condensador (en V)?",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="600px"))
qB3 = widgets.RadioButtons(
    options=["La corriente y el voltaje del C suben juntos",
             "Mientras V_C sube, la corriente baja",
             "La corriente se queda constante mientras V_C sube"],
    description="B3 — Durante la carga de un condensador:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)
submitB = widgets.Button(description="Enviar respuestas Parte B",
                         button_style="success",
                         layout=widgets.Layout(width="280px"))
feedB = widgets.HTML()

def on_submitB(b):
    # τ = R·C = 2000·100e-6 = 0.2 s = 200 ms
    b1_ok = abs(qB1.value - 200.0) < 10.0
    # Después de 5τ → 99% de V = 9.9 V
    b2_ok = abs(qB2.value - 9.9) < 0.5
    b3_ok = qB3.value == "Mientras V_C sube, la corriente baja"

    TR.answer("B", "qB1", qB1.value, correct=b1_ok, expected=200.0)
    TR.answer("B", "qB2", qB2.value, correct=b2_ok, expected=9.9)
    TR.answer("B", "qB3", qB3.value, correct=b3_ok,
              expected="Mientras V_C sube, la corriente baja")

    score = sum([b1_ok, b2_ok, b3_ok]) / 3 * 100
    TR.outcome("B", score, completed=True)

    feedB.value = f"""
    <div style="background:#EFF6FF; border-left:4px solid #1D4ED8;
                padding:14px; margin-top:10px; border-radius:6px;">
        <h4 style="margin:0; color:#0369A1;">Puntaje Parte B: {score:.0f}/100</h4>
        <ul style="margin:8px 0; color:#333;">
            <li>B1 — τ esperado = 200 ms (= 2000Ω · 100μF)  {('✅' if b1_ok else '❌')}</li>
            <li>B2 — V_C tras 5τ ≈ 9.9 V (99% de V)  {('✅' if b2_ok else '❌')}</li>
            <li>B3 — {('✅' if b3_ok else '❌ — V_C y corriente son curvas espejo')}</li>
        </ul>
    </div>
    """
    submitB.disabled = True

submitB.on_click(on_submitB)
display(widgets.VBox([qB1, qB2, qB3, submitB, feedB]))


/tmp/ipykernel_6642/3682259062.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":  datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_6642/3682259062.py:116: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":    datetime.datetime.utcnow().isoformat(),


---

## 🟣 Parte C — Circuito RC: descarga del condensador

### 📚 Concepto

Imagina que ya cargaste el condensador a un voltaje inicial **V₀**, y ahora **desconectas la fuente** y dejas que el condensador se descargue a través del resistor. ¿Qué sucede?

El voltaje del condensador **decrece exponencialmente** desde V₀ hasta cero:

$$ V_C(t) = V_0 \cdot e^{-t / \tau} $$

La constante de tiempo es la misma que en la carga:

$$ \tau = R \cdot C $$

### 🔑 Detalle importante

En la descarga, el momento clave es cuando V_C llega al **37% de V₀** (en lugar del 63% de la carga). Eso ocurre exactamente en t = τ.

> 💡 **Reto:** mueve el slider de tiempo hasta encontrar el instante donde V_C = 37% de V₀. Compara ese tiempo con la fórmula τ = R·C. Deberían coincidir.


In [7]:
# ═══ Simulador Parte C — Circuito RC descarga ═══
ensure_runtime()  # Verifica que la configuración esté lista


sliderC_V0 = widgets.FloatSlider(value=10.0, min=1, max=24, step=0.5,
                                 description="V₀ (V):", readout_format=".1f",
                                 style={"description_width": "60px"},
                                 layout=widgets.Layout(width="320px"))
sliderC_R = widgets.FloatSlider(value=1000, min=100, max=10000, step=100,
                                description="R (Ω):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))
sliderC_C = widgets.FloatSlider(value=50.0, min=1, max=500, step=1,
                                description="C (μF):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="320px"))

sliderC_t = widgets.FloatSlider(value=50.0, min=0.0, max=500.0, step=1.0,
                                description="t (ms):", readout_format=".0f",
                                style={"description_width": "60px"},
                                layout=widgets.Layout(width="660px"))

metricsC = widgets.HTML()
plotC    = widgets.Output()


def draw_circuit_RC_discharge(V0, R, C, ax):
    """Circuito RC en descarga: condensador (con V₀) y resistor en lazo cerrado.
       NO hay fuente porque ya se desconectó."""
    ax.clear()
    ax.set_xlim(0, 12); ax.set_ylim(0, 7)
    ax.set_aspect("equal"); ax.axis("off")

    # Lazo: condensador a la izquierda, resistor a la derecha
    ax.plot([2, 2], [2.6, 5.5], color="#333", lw=2)        # vertical izquierdo (C)
    ax.plot([2, 5], [5.5, 5.5], color="#333", lw=2)        # superior izquierdo
    ax.plot([7, 10], [5.5, 5.5], color="#333", lw=2)       # superior derecho
    ax.plot([10, 10], [2, 5.5], color="#333", lw=2)        # vertical derecho
    ax.plot([2, 10], [2, 2], color="#333", lw=2)           # inferior
    ax.plot([2, 2], [2, 2.4], color="#333", lw=2)          # cable inferior C

    # Condensador (con V₀ inicial indicado)
    draw_capacitor_v(ax, 2, 2.5, "C", f"{C:.0f}μF", "#7C3AED")
    ax.text(0.7, 2.5, f"V₀={V0:.1f}V",
            ha="center", va="center", fontsize=9, fontweight="bold",
            color="#1D4ED8",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor="#1D4ED8"))

    # Resistor R en el cable superior
    draw_resistor_h(ax, 5, 5.5, 2, "R", f"{R:.0f}Ω", "#C2410C", label_above=True)

    # Indicador "sin fuente"
    ax.text(6, 1.4, "(sin fuente — el condensador se descarga sobre R)",
            ha="center", fontsize=8, fontstyle="italic", color="#666")

    # Flecha de corriente
    ax.annotate("", xy=(4, 5.5), xytext=(3, 5.5),
                arrowprops=dict(arrowstyle="->", color="#15803D", lw=2))
    ax.text(3.5, 5.95, "i(t)", ha="center",
            fontsize=10, fontweight="bold", color="#15803D")


def update_C(change=None):
    V0 = sliderC_V0.value
    R = sliderC_R.value
    C_uF = sliderC_C.value
    C_F  = C_uF * 1e-6
    t_query_ms = sliderC_t.value
    t_query_s  = t_query_ms / 1000.0

    tau_s  = R * C_F
    tau_ms = tau_s * 1000

    Vc_query = V0 * np.exp(-t_query_s / tau_s)
    pct_remaining = (Vc_query / V0 * 100) if V0 > 0 else 0

    # Reto: leer τ → si Vc_query está cerca del 37% de V0, está leyendo bien τ
    distance_to_37 = abs(Vc_query - 0.368 * V0)

    if change is not None and change.get("type") == "change" and change.get("name") == "value":
        widget_desc = change.owner.description.replace(":", "").strip()
        if widget_desc.startswith("t"):
            TR.widget_event(part="C", widget_name="t (slider tiempo)",
                            old=change["old"], new=change["new"],
                            target=tau_ms)
        else:
            TR.widget_event(part="C", widget_name=widget_desc,
                            old=change["old"], new=change["new"])
            TR.simulation(
                part="C",
                params={"V0": V0, "R": R, "C_uF": C_uF},
                results={"tau_ms": round(tau_ms, 3),
                         "Vc_at_tau": round(0.368 * V0, 3)},
            )

    target_msg = ""
    if distance_to_37 < 0.1 * V0:  # tolerancia 10% de V0
        target_msg = status_badge(
            f"🎯 ¡Lectura correcta de τ! Estás en t={t_query_ms:.1f}ms, "
            f"V_C={Vc_query:.2f}V (37% de V₀)", "success"
        )

    metricsC.value = (
        metric_card("V₀ inicial", f"{V0:.1f}", "V", "#1D4ED8") +
        metric_card("τ = R·C (teórico)", f"{tau_ms:.2f}", "ms", "#C2410C") +
        metric_card("V₀ × 37% (objetivo)", f"{0.368*V0:.2f}", "V", "#7C3AED") +
        metric_card(f"V_C(t={t_query_ms:.0f}ms)", f"{Vc_query:.2f}", f"V ({pct_remaining:.0f}% V₀)", "#15803D") +
        f"<div style='margin-top: 8px'>{target_msg}</div>"
    )

    with plotC:
        clear_output(wait=True)
        fig = plt.figure(figsize=(12, 6))
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.4], wspace=0.15)
        ax_circ = fig.add_subplot(gs[0])
        ax_curve = fig.add_subplot(gs[1])

        draw_circuit_RC_discharge(V0, R, C_uF, ax_circ)

        # Curva V_C(t)
        t_max_plot_s = max(6 * tau_s, t_query_s * 1.2, 0.001)
        t_arr = np.linspace(0, t_max_plot_s, 500)
        Vc_arr = V0 * np.exp(-t_arr / tau_s)
        t_arr_ms = t_arr * 1000

        ax_curve.plot(t_arr_ms, Vc_arr, color="#7C3AED", lw=2.5, label="V_C(t)")
        ax_curve.axhline(V0, color="#1D4ED8", ls="--", lw=1.2,
                         label=f"V₀ = {V0:.1f} V")
        ax_curve.axhline(0.368 * V0, color="#C2410C", ls=":", lw=1.5,
                         label=f"37% V₀ = {0.368*V0:.2f} V")
        # Marcador en τ
        ax_curve.axvline(tau_ms, color="#C2410C", ls=":", lw=1.2)
        ax_curve.text(tau_ms, V0 * 0.85, f"τ={tau_ms:.1f}ms",
                      color="#C2410C", fontsize=9, fontweight="bold", ha="left")

        # Marcador del slider
        ax_curve.axvline(t_query_ms, color="#B91C1C", ls="-", lw=1.0, alpha=0.6)
        ax_curve.scatter([t_query_ms], [Vc_query], color="#B91C1C", s=90,
                         zorder=5, edgecolors="white", linewidths=2)
        ax_curve.annotate(f"t={t_query_ms:.0f}ms\nV_C={Vc_query:.2f}V",
                          xy=(t_query_ms, Vc_query),
                          xytext=(15, 15), textcoords="offset points",
                          fontsize=9, fontweight="bold", color="#B91C1C",
                          bbox=dict(boxstyle="round,pad=0.3",
                                    facecolor="#FEF2F2", edgecolor="#B91C1C"))

        ax_curve.set_xlabel("Tiempo t (ms)")
        ax_curve.set_ylabel("V_C (V)")
        ax_curve.set_title("Descarga exponencial del condensador")
        ax_curve.legend(loc="upper right", fontsize=9)
        ax_curve.grid(True, alpha=0.3)
        ax_curve.set_xlim(0, t_max_plot_s * 1000)
        ax_curve.set_ylim(0, V0 * 1.15)
        plt.show()


sliderC_V0.observe(update_C, names="value")
sliderC_R.observe(update_C, names="value")
sliderC_C.observe(update_C, names="value")
sliderC_t.observe(update_C, names="value")
update_C()

display(widgets.VBox([
    widgets.HBox([sliderC_V0, sliderC_R, sliderC_C]),
    sliderC_t,
    metricsC,
    plotC,
]))


In [8]:
# Preguntas Parte C
qC1 = widgets.FloatText(description="C1 — Si V₀=20V, R=500Ω, C=200μF, ¿cuánto vale τ (en ms)?",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="600px"))
qC2 = widgets.FloatText(description="C2 — Después de un tiempo t=τ, ¿qué voltaje queda en el condensador (en V)?",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="600px"))
qC3 = widgets.RadioButtons(
    options=["τ es distinto en carga y descarga",
             "τ es exactamente el mismo (R·C) para ambos casos",
             "En descarga τ es la mitad que en carga"],
    description="C3 — La constante de tiempo τ:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)
submitC = widgets.Button(description="Enviar respuestas Parte C",
                         button_style="success",
                         layout=widgets.Layout(width="280px"))
feedC = widgets.HTML()

def on_submitC(b):
    # τ = R·C = 500·200e-6 = 0.1 s = 100 ms
    c1_ok = abs(qC1.value - 100.0) < 5.0
    # V_C en t=τ → 37% de 20 = 7.36 V
    c2_ok = abs(qC2.value - 7.36) < 0.5
    c3_ok = qC3.value == "τ es exactamente el mismo (R·C) para ambos casos"

    TR.answer("C", "qC1", qC1.value, correct=c1_ok, expected=100.0)
    TR.answer("C", "qC2", qC2.value, correct=c2_ok, expected=7.36)
    TR.answer("C", "qC3", qC3.value, correct=c3_ok,
              expected="τ es exactamente el mismo (R·C) para ambos casos")

    score = sum([c1_ok, c2_ok, c3_ok]) / 3 * 100
    TR.outcome("C", score, completed=True)

    feedC.value = f"""
    <div style="background:#FAF5FF; border-left:4px solid #7C3AED;
                padding:14px; margin-top:10px; border-radius:6px;">
        <h4 style="margin:0; color:#5B21B6;">Puntaje Parte C: {score:.0f}/100</h4>
        <ul style="margin:8px 0; color:#333;">
            <li>C1 — τ esperado = 100 ms (= 500Ω · 200μF)  {('✅' if c1_ok else '❌')}</li>
            <li>C2 — V_C(τ) = 0.368 · 20 ≈ 7.36 V  {('✅' if c2_ok else '❌')}</li>
            <li>C3 — {('✅' if c3_ok else '❌ — τ depende solo de R y C, no del proceso')}</li>
        </ul>
    </div>
    """
    submitC.disabled = True

submitC.on_click(on_submitC)
display(widgets.VBox([qC1, qC2, qC3, submitC, feedC]))


/tmp/ipykernel_6642/3682259062.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":  datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_6642/3682259062.py:116: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":    datetime.datetime.utcnow().isoformat(),


---

## 📤 Paso final — Resumen y envío de datos

Cuando hayas completado las **3 partes**, ejecuta las siguientes celdas para enviar tus datos al docente.


In [9]:
# Resumen visual
def render_summary():
    s = TR.summary()
    rows = ""
    for o in TR.outcomes:
        color = "#15803D" if o["score"] >= 70 else "#D97706" if o["score"] >= 40 else "#B91C1C"
        rows += f"""
        <tr>
            <td style="padding:8px 12px; font-weight:600">Parte {o['part']}</td>
            <td style="padding:8px 12px; color:{color}; font-weight:700">{o['score']:.0f}/100</td>
            <td style="padding:8px 12px; text-align:center">{o['n_widget_events']}</td>
            <td style="padding:8px 12px; text-align:center">{o['n_simulations']}</td>
            <td style="padding:8px 12px; text-align:right">{o['duration_sec']:.0f} s</td>
        </tr>
        """
    html = f"""
    <div style="background:white; border:1px solid #E5E7EB; border-radius:8px;
                padding:20px; max-width:700px; box-shadow:0 1px 3px rgba(0,0,0,0.1);">
        <h3 style="margin:0 0 14px 0; color:#1D4ED8;">📊 Resumen — Actividad 2</h3>
        <div style="display:flex; gap:10px; flex-wrap:wrap; margin-bottom:14px;">
            {metric_card("Estudiante", TR.est_id or "(no identificado)", "", "#666666")}
            {metric_card("Duración", f"{s['duration_min']}", "min", "#1D4ED8")}
            {metric_card("Eventos", f"{s['n_widget_events']}", "sliders", "#7C3AED")}
            {metric_card("Simulaciones", f"{s['n_simulations']}", "cálculos", "#0F766E")}
            {metric_card("Partes", f"{s['n_outcomes']}/3", "completadas", "#15803D")}
        </div>
        <table style="width:100%; border-collapse: collapse; font-size:13px;">
            <thead>
                <tr style="background:#F3F4F6;">
                    <th style="padding:8px 12px; text-align:left">Parte</th>
                    <th style="padding:8px 12px; text-align:left">Puntaje</th>
                    <th style="padding:8px 12px; text-align:center">Eventos</th>
                    <th style="padding:8px 12px; text-align:center">Simulaciones</th>
                    <th style="padding:8px 12px; text-align:right">Duración</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
    </div>
    """
    display(HTML(html))


render_summary()


Parte,Puntaje,Eventos,Simulaciones,Duración
Parte A,100/100,15,15,1083 s
Parte B,100/100,0,0,1489 s
Parte C,100/100,0,0,2029 s


In [ ]:
# ═══ Envío automático a Google Sheets ═══
GSHEETS_URL = "https://docs.google.com/spreadsheets/d/1tnjGuahIoZfYZfHHNbD9cG58NP3-0NOKQHqrSbxCtBs/edit?usp=sharing"

SHEETS_SCHEMA = {
    "sessions": [
        "session_id", "est_id", "activity",
        "timestamp_utc", "duration_min", "parts_completed",
    ],
    "events": [
        "session_id", "est_id", "activity", "type", "part",
        "widget", "old_value", "new_value", "target", "distance_to_target",
        "params", "results", "error_pct",
        "t_relative", "gap_since_last", "timestamp",
    ],
    "answers": [
        "session_id", "est_id", "activity", "part", "qid",
        "value", "expected", "is_correct", "t_relative", "timestamp",
    ],
    "outcomes": [
        "session_id", "est_id", "activity", "part",
        "score", "completed", "duration_sec",
        "n_widget_events", "n_simulations", "timestamp",
    ],
}


def ensure_worksheet(sh, name, headers):
    try:
        ws = sh.worksheet(name)
        first_row = ws.row_values(1)
        if not first_row:
            ws.update("A1", [headers])
        return ws
    except Exception:
        ws = sh.add_worksheet(title=name, rows=1000, cols=len(headers))
        ws.update("A1", [headers])
        return ws


def push_to_sheets(url):
    if "PEGAR_AQUI" in url or not url:
        display(HTML("""
        <div style="background:#FEF3C7; border-left:4px solid #D97706;
                    padding:14px; border-radius:6px;">
            <strong style="color:#92400E">⚠️ La URL de Google Sheets no está configurada.</strong>
        </div>
        """))
        return False

    try:
        from google.colab import auth
        import gspread
        from google.auth import default
    except ImportError as e:
        display(HTML(f"""
        <div style="background:#FEE2E2; border-left:4px solid #B91C1C;
                    padding:14px; border-radius:6px;">
            <strong style="color:#7F1D1D">⚠️ Faltan librerías de Colab:</strong>
            <code>{e}</code>
        </div>
        """))
        return False

    try:
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
    except Exception as e:
        display(HTML(f"""<div style="background:#FEE2E2; border-left:4px solid #B91C1C;
                    padding:14px; border-radius:6px;">
            <strong style="color:#7F1D1D">⚠️ Error de autenticación:</strong>
            <code>{e}</code></div>"""))
        return False

    try:
        sh = gc.open_by_url(url)
    except Exception as e:
        display(HTML(f"""<div style="background:#FEE2E2; border-left:4px solid #B91C1C;
                    padding:14px; border-radius:6px;">
            <strong style="color:#7F1D1D">⚠️ No se pudo abrir la hoja:</strong>
            <code>{e}</code></div>"""))
        return False

    try:
        ws_sessions = ensure_worksheet(sh, "sessions", SHEETS_SCHEMA["sessions"])
        ws_events   = ensure_worksheet(sh, "events",   SHEETS_SCHEMA["events"])
        ws_answers  = ensure_worksheet(sh, "answers",  SHEETS_SCHEMA["answers"])
        ws_outcomes = ensure_worksheet(sh, "outcomes", SHEETS_SCHEMA["outcomes"])
    except Exception as e:
        display(HTML(f"""<div style="background:#FEE2E2; border-left:4px solid #B91C1C;
                    padding:14px; border-radius:6px;">
            <strong style="color:#7F1D1D">⚠️ Error preparando pestañas:</strong>
            <code>{e}</code></div>"""))
        return False

    try:
        ws_sessions.append_row([
            TR.session_id, TR.est_id or "", TR.activity,
            datetime.datetime.utcnow().isoformat(),
            round((time.time() - TR.t_start) / 60, 2),
            len(TR.outcomes),
        ])
        if TR.events:
            rows_events = [[str(ev.get(k, "")) for k in SHEETS_SCHEMA["events"]]
                           for ev in TR.events]
            ws_events.append_rows(rows_events, value_input_option="RAW")
        if TR.answers:
            rows_answers = [[str(a.get(k, "")) for k in SHEETS_SCHEMA["answers"]]
                            for a in TR.answers]
            ws_answers.append_rows(rows_answers, value_input_option="RAW")
        if TR.outcomes:
            rows_outcomes = [[str(o.get(k, "")) for k in SHEETS_SCHEMA["outcomes"]]
                             for o in TR.outcomes]
            ws_outcomes.append_rows(rows_outcomes, value_input_option="RAW")
    except Exception as e:
        display(HTML(f"""<div style="background:#FEE2E2; border-left:4px solid #B91C1C;
                    padding:14px; border-radius:6px;">
            <strong style="color:#7F1D1D">⚠️ Error escribiendo en la hoja:</strong>
            <code>{e}</code></div>"""))
        return False

    display(HTML(f"""
    <div style="background:#DCFCE7; border-left:4px solid #15803D;
                padding:14px; border-radius:6px;">
        <strong style="color:#14532D">✅ Datos de Actividad 2 enviados correctamente.</strong>
        <ul style="margin:8px 0 0 0; color:#14532D; font-size:13px;">
            <li>1 sesión registrada</li>
            <li>{len(TR.events)} eventos</li>
            <li>{len(TR.answers)} respuestas</li>
            <li>{len(TR.outcomes)} partes completadas</li>
        </ul>
    </div>
    """))
    return True


sent = push_to_sheets(GSHEETS_URL)

if not sent:
    backup = {
        "session": {
            "session_id":   TR.session_id,
            "est_id":       TR.est_id,
            "activity":     TR.activity,
            "duration_min": round((time.time() - TR.t_start) / 60, 2),
        },
        "events":   TR.events,
        "answers":  TR.answers,
        "outcomes": TR.outcomes,
    }
    print("─" * 65)
    print("  RESPALDO MANUAL EN JSON")
    print("─" * 65)
    print(json.dumps(backup, indent=2, ensure_ascii=False))


---

## ✅ ¡Has terminado la Actividad 2!

> 📚 **Conceptos que dominas ahora:**
> - El **régimen transitorio**: corrientes y voltajes que evolucionan en el tiempo siguiendo curvas exponenciales.
> - La **constante de tiempo τ**: cuán rápido el sistema alcanza el estado estable.
>   - En RL: τ = L / R
>   - En RC: τ = R · C
> - La **corriente límite** en un circuito RL: I_max = V / R.
> - La **carga** y **descarga** de un condensador, y la simetría matemática entre ambos procesos.
> - Que el **63% (carga) y 37% (descarga)** son los porcentajes universales en t = τ.

En la **Actividad 3** vamos a ir un paso más allá: en vez de una fuente DC que se enciende una sola vez, usaremos una **fuente AC** (corriente alterna) que cambia de signo continuamente, y verás cómo las bobinas y los condensadores reaccionan a esa alternancia con un fenómeno llamado **impedancia**.

¡Nos vemos en la **Actividad 3**!
